In [ ]:
!pip install ultralytics huggingface_hub datasets -q
!pip install torch torchvision torchaudio -q
!pip install matplotlib seaborn pandas numpy -q
!pip install timm einops -q  # for transformer blocks
print("✅ All libraries installed")

In [ ]:
from huggingface_hub import snapshot_download
import os

# Download entire dataset
dataset_path = snapshot_download(
    repo_id="KaraAgroAI/CADI-AI",
    repo_type="dataset",
    local_dir="./cadi_ai_dataset"
)
print(f"✅ Dataset downloaded to: {dataset_path}")

# Check structure
for root, dirs, files in os.walk("./cadi_ai_dataset"):
    level = root.replace("./cadi_ai_dataset", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"{indent}  {f}")

In [ ]:
import os

# Walk the whole downloaded folder and print everything
for root, dirs, files in os.walk("./cadi_ai_dataset"):
    # Skip hidden folders
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace("./cadi_ai_dataset", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        subindent = "  " * (level + 1)
        for f in files[:5]:   # show first 5 files per folder
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... ({len(files)} total)")

In [ ]:
# Check if git-lfs is available (needed for HF image datasets)
!apt-get install git-lfs -q
!git lfs install
!pip install huggingface_hub datasets -q -U
print("✅ Done")

In [ ]:
from huggingface_hub import HfFileSystem, hf_hub_download
import os

hffs = HfFileSystem()

# List what's actually in the repo
print("Files in KaraAgroAI/CADI-AI:")
files = hffs.ls("datasets/KaraAgroAI/CADI-AI", detail=False)
for f in files:
    print(" ", f)


In [ ]:
from huggingface_hub import HfFileSystem

hffs = HfFileSystem()

# Find ALL files inside Data folder recursively
all_files = hffs.find(
    "datasets/KaraAgroAI/CADI-AI/Data",
    maxdepth=4,
    withdirs=True,
    detail=False
)

for f in all_files[:40]:   # show first 40 entries
    print(f)

print(f"\nTotal entries: {len(all_files)}")

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs("./cadi_ai_dataset", exist_ok=True)

# Download all 3 zips
for split in ["train", "val", "test"]:
    print(f"Downloading {split}.zip ...")
    path = hf_hub_download(
        repo_id="KaraAgroAI/CADI-AI",
        repo_type="dataset",
        filename=f"Data/{split}.zip",
        local_dir="./cadi_ai_dataset"
    )
    print(f"  ✅ Saved to: {path}")

In [ ]:
import zipfile, os

for split in ["train", "val", "test"]:
    zip_path = f"./cadi_ai_dataset/Data/{split}.zip"
    print(f"Unzipping {split}.zip ...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall("./cadi_ai_dataset/Data/")
    print(f"  ✅ Done")

print("\nAll unzipped!")

In [ ]:
import os

for root, dirs, files in os.walk("./cadi_ai_dataset/Data"):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace("./cadi_ai_dataset/Data", "").count(os.sep)
    if level < 4:
        img_count = len([f for f in files if f.lower().endswith(('.jpg','.png','.jpeg'))])
        txt_count = len([f for f in files if f.endswith('.txt')])
        print("  "*level + os.path.basename(root) + f"/   imgs:{img_count}  labels:{txt_count}")

In [ ]:
import os, glob
from collections import Counter

# ✅ CORRECT paths — data is already here
BASE = "./cadi_ai_dataset/Data"
CLASS_NAMES = ["abiotic", "disease", "insect"]

# Count images and labels
for split in ["train", "val", "test"]:
    imgs = (glob.glob(f"{BASE}/{split}/images/*.jpg") +
            glob.glob(f"{BASE}/{split}/images/*.jpeg") +
            glob.glob(f"{BASE}/{split}/images/*.png"))
    lbls = glob.glob(f"{BASE}/{split}/labels/*.txt")
    print(f"{split:5s}: {len(imgs):5d} images | {len(lbls):5d} labels")

# Class distribution from train labels
print("\nClass distribution (train):")
class_counts = Counter()
for lbl_path in glob.glob(f"{BASE}/train/labels/*.txt"):
    with open(lbl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                cls = int(line.split()[0])
                class_counts[cls] += 1

for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name:10s}: {class_counts[cls_id]:5d} objects")

In [ ]:
import glob, random, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

BASE = "./cadi_ai_dataset/Data"
CLASS_NAMES = ["abiotic", "disease", "insect"]
COLORS = ["#3B8BD4", "#E24B4A", "#1D9E75"]  # blue, red, green

def show_samples(split, n=6):
    imgs = (glob.glob(f"{BASE}/{split}/images/*.jpg") +
            glob.glob(f"{BASE}/{split}/images/*.jpeg") +
            glob.glob(f"{BASE}/{split}/images/*.png"))
    sample = random.sample(imgs, min(n, len(imgs)))

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f"CADI-AI — {split} samples with YOLO annotations", fontsize=13)

    for ax, img_path in zip(axes.flatten(), sample):
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        ax.imshow(img)

        lbl_path = img_path.replace("images", "labels")
        lbl_path = os.path.splitext(lbl_path)[0] + ".txt"

        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    cls = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = (cx - bw/2) * W
                    y1 = (cy - bh/2) * H
                    rect = patches.Rectangle(
                        (x1, y1), bw*W, bh*H,
                        linewidth=2, edgecolor=COLORS[cls], facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.text(x1, y1 - 6, CLASS_NAMES[cls],
                            color=COLORS[cls], fontsize=8, fontweight='bold',
                            bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        ax.axis('off')
        ax.set_title(os.path.basename(img_path), fontsize=7)

    plt.tight_layout()
    plt.savefig(f'./sample_{split}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved sample_{split}.png")

show_samples("train")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CLASS_NAMES = ["abiotic", "disease", "insect"]
counts = [1285, 11370, 5626]   # your actual numbers from Step 5
COLORS = ["#3B8BD4", "#E24B4A", "#1D9E75"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(CLASS_NAMES, counts, color=COLORS, edgecolor='white', linewidth=0.8)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 150,
            f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title("CADI-AI Class Distribution (Train Set)", fontsize=13)
ax.set_ylabel("Number of Annotated Objects")
ax.set_ylim(0, max(counts) * 1.15)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('./class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved class_distribution.png")

In [ ]:
import yaml, os

config = {
    'path': os.path.abspath('./cadi_ai_dataset/Data'),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 3,
    'names': ['abiotic', 'disease', 'insect']
}

with open('./cadi_ai.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✅ YAML config created:")
print(open('./cadi_ai.yaml').read())

In [ ]:
import yaml, os, glob, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter

BASE = "./cadi_ai_dataset/Data"
CLASS_NAMES = ["abiotic", "disease", "insect"]
COLORS = ["#3B8BD4", "#E24B4A", "#1D9E75"]

# Create YAML
config = {
    'path': os.path.abspath(BASE),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 3,
    'names': CLASS_NAMES
}
with open('./cadi_ai.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print("✅ YAML created:")
print(open('./cadi_ai.yaml').read())

In [ ]:
# --- Sample images with boxes ---
imgs = glob.glob(f"{BASE}/train/images/*.jpg")
sample = random.sample(imgs, min(6, len(imgs)))
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("CADI-AI Train Samples with YOLO Annotations", fontsize=13)
for ax, img_path in zip(axes.flatten(), sample):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    ax.imshow(img)
    lbl = os.path.splitext(img_path.replace("images","labels"))[0] + ".txt"
    if os.path.exists(lbl):
        for line in open(lbl):
            p = line.strip().split()
            if not p: continue
            c, cx, cy, bw, bh = int(p[0]), *map(float,p[1:5])
            x1,y1 = (cx-bw/2)*W, (cy-bh/2)*H
            ax.add_patch(patches.Rectangle((x1,y1),bw*W,bh*H,
                linewidth=2,edgecolor=COLORS[c],facecolor='none'))
            ax.text(x1,y1-6,CLASS_NAMES[c],color=COLORS[c],fontsize=8,
                fontweight='bold',bbox=dict(facecolor='white',alpha=0.5,pad=1,edgecolor='none'))
    ax.axis('off')
plt.tight_layout()
plt.savefig('./sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

# --- Class distribution ---
counts = [1285, 11370, 5626]
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(CLASS_NAMES, counts, color=COLORS, edgecolor='white')
for bar,cnt in zip(bars,counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+150,
            f'{cnt:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title("Class Distribution — Train Set"); ax.set_ylabel("Objects")
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('./class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved sample_images.png and class_distribution.png")

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

print("🚀 Training BASELINE YOLOv8n ...")
baseline = YOLO('yolov8n.pt')
baseline.train(
    data='./cadi_ai.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='baseline_yolov8n',
    project='./runs',
    patience=10,
    lr0=0.01,
    verbose=False   # keep output clean
)
print("✅ Baseline training done!")

In [ ]:
from ultralytics import YOLO

print("🚀 Training MODIFIED YOLOv8m ...")
modified = YOLO('yolov8m.pt')
modified.train(
    data='./cadi_ai.yaml',
    epochs=40,
    imgsz=640,
    batch=8,
    name='modified_yolov8m',
    project='./runs',
    patience=15,
    lr0=0.005,
    lrf=0.01,
    # ★ Novel modifications — augmentation strategy
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    degrees=10.0,
    # ★ Novel modifications — loss weights
    box=7.5,
    cls=0.5,
    dfl=1.5,
    verbose=False
)
print("✅ Modified training done!")